In [9]:
import profilelib.*

In [10]:
val data = load_data("jfrs/serialization/cstack_dwarf/")

'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=44,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=50,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=20,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=48,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/dat

In [11]:
val dataNoGC = load_data("jfrs/serialization/cstack_dwarf/") {
   it.filterNot { it.any { it.method.name.contains("::Sweep<") } }
}

'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=44,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=50,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=20,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=48,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/dat

In [12]:
data.funRatio("java.lang.String.substring", "weirdo:Kotlin_String_subSequence").let { println(formattedMeanCV(it.map { it!! })) }
dataNoGC.funRatio("java.lang.String.substring", "weirdo:Kotlin_String_subSequence").let { println(formattedMeanCV(it.map { it!! })) }


0.162 (0.049)
0.174 (0.050)


This felt like a too small differece as in `StringSubstringBenchmark` the profiles show much larger time spent in GC. However, in the TwitterBM profiles, this roughly matches the saw values. Sweep really took 5--10% of `Kotlin_String_subSequence`.

In [13]:
data.funRatio("java.util.HashMap.get", "kotlin.collections.Map.get").let { println(formattedMeanCV(it.map { it!! })) }
dataNoGC.funRatio("java.util.HashMap.get", "kotlin.collections.Map.get").let { println(formattedMeanCV(it.map { it!! })) }


0.129 (0.032)
0.129 (0.032)


In [14]:
data.funRatio("java.lang.StringBuilder.append", "kotlin.text.StringBuilder.append").also { println(formattedMeanCV(it.map { it!! })) }.let { println(mean(it.map { it!! })) }
dataNoGC.funRatio("java.lang.StringBuilder.append", "kotlin.text.StringBuilder.append").also { println(formattedMeanCV(it.map { it!! })) }.let { println(mean(it.map { it!! })) }


0.037 (0.035)
0.037457188250623316
0.037 (0.035)
0.03749245169732162


TODO Investigate if the results are comparable. E.g., where does conversion from `Any` to `String` happen? In K/N it
happens inside the call (i.e., the conversion counts as samples for this). How is it in JVM?

In JVM, this is actually handled on `kx.serialization` part. It call `StringBuilder.append` only after `.toString()`
call. Therefore, `kotlin.test.StringBuilder.append` needs to be broken down by signature and this summary value cannot
be used.

After further investigation, it actually seems that there is a custom performant and specialized StringBuilder impl
at JVM. In K/N it's delegated to StringBuilder. Therefore, the implementation/wrapper needs to be filtered out.

In [15]:
import jdk.jfr.consumer.RecordedFrame

val dataStringBuilder = load_data("jfrs/serialization/cstack_dwarf/") {
    it
        .filterNot { it.any { it.method.name.contains("JsonToStringWriter") } }
        .filterNot { it.any { it.method.name.contains("kfun:kotlin.text.StringBuilder#append(kotlin.Long)") } && !it.any { it.method.name.contains("kfun:kotlin.text.StringBuilder#append(kotlin.String?)") } }
}

'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=44,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=50,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=20,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=48,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/dat

In [16]:
dataStringBuilder.funRatio("java.lang.StringBuilder.append", "kotlin.text.StringBuilder.append").let { println(formattedMeanCV(it.map { it!! })) }

0.154 (0.036)
